# Auditoria de hiperparâmetros (remoto)



In [2]:
import json
import os
from pathlib import Path

import pandas as pd
import requests
import yaml
from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv()

HF_ENDPOINT = "https://huggingface.co".rstrip("/")
# HF_TOKEN = os.environ.get("HF_TOKEN")

REPO_ID = "anon-review-12345/dora-reproducibility-study"
REVISION = "main"

RUNS = {
    "lora-v3":           "lora/seed-42",
    "lora-match-seed42": "lora-lr-match/seed-42",
    "lora-match-seed43": "lora-lr-match/seed-43",
    "dora-seed42":       "dora/seed-42",
    "dora-seed43":       "dora/seed-43",
    "qdora-seed42-fp32": "qdora-fp32/seed-42",
    "qdora-seed43-fp32": "qdora-fp32/seed-43",
    "qlora-seed42":      "qlora/seed-42",
    "qlora-seed43":      "qlora/seed-43",
}

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Acesso remoto

As mesmas primitivas de `mechanistic_analysis_remote.ipynb`: só JSON e YAML
pequenos são lidos aqui, nada é gravado em disco além das tabelas exportadas.

In [3]:
SESSION = requests.Session()
# if HF_TOKEN:
#     SESSION.headers["Authorization"] = f"Bearer {HF_TOKEN}"

REQUEST_TIMEOUT = 60
REQUEST_RETRIES = 3


def http_get(repo_id, path, revision=REVISION, headers=None):
    url = f"{HF_ENDPOINT}/{repo_id}/resolve/{revision}/{path}"
    last_error = None
    for _ in range(REQUEST_RETRIES):
        try:
            response = SESSION.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response
        except requests.RequestException as error:
            last_error = error
    raise RuntimeError(f"GET failed after {REQUEST_RETRIES} attempts: {url}") from last_error


def read_remote_json(repo_id, path, revision=REVISION):
    return json.loads(http_get(repo_id, path, revision).text)


def read_remote_yaml(repo_id, path, revision=REVISION):
    return yaml.safe_load(http_get(repo_id, path, revision).text)


HF_API = HfApi()

try:
    REPO_FILES = set(HF_API.list_repo_files(REPO_ID, revision=REVISION))
except Exception as error:
    raise RuntimeError(
        f"Could not list {REPO_ID}@{REVISION}. Gated repositories need a valid "
        "HF_TOKEN in the environment or in .env."
    ) from error

print(f"{REPO_ID}@{REVISION}: {len(REPO_FILES):,} files listed (no download yet)")

anon-review-12345/dora-reproducibility-study@main: 797 files listed (no download yet)


In [4]:
(OUTPUT_DIR / "final_path_versions.txt").write_text(
    json.dumps([[f"{REPO_ID}/{repo_dir}" for repo_dir in RUNS.values()]])
)

583

### Train

In [5]:
full = {}
parameters = {}
devices = {}
adapter = {}
quant = {}

for run_name, repo_dir in RUNS.items():
    metadata = read_remote_json(REPO_ID, f"{repo_dir}/run_metadata.json")
    full[run_name] = metadata
    parameters[run_name] = metadata["training"]
    devices[run_name] = metadata["cuda_devices"][0]
    adapter[run_name] = metadata["adapter"]
    quant[run_name] = metadata["quantization"]

df = pd.DataFrame(full).sort_index()
df_params = pd.DataFrame(parameters).sort_index()
df_params = df_params.loc[df_params.index != "resume_from_checkpoint", :]
df_devices = pd.DataFrame(devices).sort_index()
df_adapter = pd.DataFrame(adapter).sort_index()
df_quant = pd.DataFrame(quant)

df

,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
adapter,"{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['..."
cuda_devices,"[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',..."
experiment,"{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '..."
model,"{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt..."
num_eval_examples,0,0,0,0,0,0,0,0,0
num_train_examples,170420,170420,170420,170420,170420,170420,170420,170420,170420
quantization,"{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf..."
total_params,8086884352,8086884352,8086884352,8087670784,8087670784,4598009856,4598009856,4597223424,4597223424
trainable_param_percent,0.700184,0.700184,0.700184,0.70984,0.70984,1.248574,1.248574,1.23168,1.23168
trainable_params,56623104,56623104,56623104,57409536,57409536,57409536,57409536,56623104,56623104


In [6]:
df_params

,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
bf16,True,True,True,True,True,True,True,True,True
dataloader_num_workers,0,0,0,0,0,0,0,0,0
dataset_text_field,text,text,text,text,text,text,text,text,text
fp16,False,False,False,False,False,False,False,False,False
gradient_accumulation_steps,1,1,1,1,1,1,1,1,1
gradient_checkpointing,True,True,True,True,True,True,True,True,True
gradient_checkpointing_kwargs,{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False}
learning_rate,0.0003,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001
logging_dir,None,None,None,None,None,None,None,None,None
logging_steps,50,50,50,50,50,50,50,50,50


In [7]:
df_devices

,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
index,0,0,0,0,0,0,0,0,0
major,9,9,9,9,9,9,9,9,9
minor,0,0,0,0,0,0,0,0,0
name,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3
total_memory_gb,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711


In [8]:
df_adapter

,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
alpha,64,64,64,64,64,64,64,64,64
bias,none,none,none,none,none,none,none,none,none
dropout,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05
magnitude_dtype,NaN,NaN,NaN,NaN,NaN,fp32,fp32,NaN,NaN
rank,32,32,32,32,32,32,32,32,32
target_modules,"[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]"
task_type,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM
use_dora,False,False,False,True,True,True,True,False,False


In [9]:
df_quant

,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
enabled,False,False,False,False,False,True,True,True,True
bits,4,4,4,4,4,4,4,4,4
quant_type,nf4,nf4,nf4,nf4,nf4,nf4,nf4,nf4,nf4
double_quant,True,True,True,True,True,True,True,True,True


In [10]:
df_adapter.to_csv(OUTPUT_DIR / "audit_adapters.csv")
df_devices.to_csv(OUTPUT_DIR / "audit_devices.csv")
df_params.to_csv(OUTPUT_DIR / "audit_params.csv")
df_quant.to_csv(OUTPUT_DIR / "audit_quantization.csv")

### Evals

O `eval_resolved_config.yaml` fica dentro de `paper_eval_results/`, direto ou em
um subdiretório `paper_eval_<timestamp>` — o padrão de nomes ordena por
recência, então o mais recente é escolhido.

In [11]:
excluded = []

full = {}
evals = {}

for run_name, repo_dir in RUNS.items():
    if run_name in excluded:
        continue

    prefix = f"{repo_dir}/paper_eval_results/"
    if f"{prefix}eval_resolved_config.yaml" in REPO_FILES:
        config_path = f"{prefix}eval_resolved_config.yaml"
    else:
        candidates = sorted({
            path[len(prefix):].split("/")[0]
            for path in REPO_FILES
            if path.startswith(prefix) and "/" in path[len(prefix):]
        })
        if not candidates:
            raise FileNotFoundError(f"Nenhum paper_eval em {REPO_ID}/{prefix}")
        config_path = f"{prefix}{candidates[-1]}/eval_resolved_config.yaml"
        if len(candidates) > 1:
            print("picked run: {}, among all {}".format(candidates[-1], ", ".join(candidates)))

    print(f"{run_name:>18} -> {config_path}")
    complete_file = read_remote_yaml(REPO_ID, config_path)
    full[run_name] = complete_file
    evals[run_name] = complete_file["eval"]

df = pd.DataFrame(full).sort_index()
df_evals = pd.DataFrame(evals).sort_index()
df_evals = df_evals.drop(index="adapter_path", errors="ignore")  # caminho local da máquina de treino

           lora-v3 -> lora/seed-42/paper_eval_results/paper_eval_20260827_103908/eval_resolved_config.yaml
picked run: paper_eval_20260717_002852, among all paper_eval_20260712_201111, paper_eval_20260717_002852
 lora-match-seed42 -> lora-lr-match/seed-42/paper_eval_results/paper_eval_20260717_002852/eval_resolved_config.yaml
 lora-match-seed43 -> lora-lr-match/seed-43/paper_eval_results/paper_eval_20260820_194824/eval_resolved_config.yaml
picked run: paper_eval_20260827_022522, among all paper_eval_20260710_192428, paper_eval_20260827_022522
       dora-seed42 -> dora/seed-42/paper_eval_results/paper_eval_20260827_022522/eval_resolved_config.yaml
       dora-seed43 -> dora/seed-43/paper_eval_results/paper_eval_20260722_130811/eval_resolved_config.yaml
 qdora-seed42-fp32 -> qdora-fp32/seed-42/paper_eval_results/paper_eval_20260825_015812/eval_resolved_config.yaml
 qdora-seed43-fp32 -> qdora-fp32/seed-43/paper_eval_results/paper_eval_20260825_015842/eval_resolved_config.yaml
      qlora

In [12]:
df_evals

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,lora-v3,lora-match-seed42,lora-match-seed43,dora-seed42,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora-seed42,qlora-seed43
allow_base_model_eval,False,False,False,False,False,False,False,False,False
batch_size,1,1,1,1,1,1,1,1,1
do_sample,False,False,False,False,False,False,False,False,False
limit,None,None,None,None,None,None,None,None,None
max_new_tokens,32,32,32,32,32,32,32,32,32
num_beams,4,4,4,4,4,4,4,4,4
save_every_n_batches,1,1,1,1,1,1,1,1,1
task_url_template,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...
tasks,"[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran..."
temperature,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1


In [13]:
df_evals.to_csv(OUTPUT_DIR / "audit_evals.csv")